In [18]:
import numpy as np
import pandas as pd
import regex 
import unicodedata
import matplotlib.pyplot as plt
import nltk

In [19]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [20]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:
total_df = pd.read_csv(r"data\all_data.csv")
total_score_df = pd.read_csv('data/all_data_scores.csv', lineterminator = "\n")
cols_sentences = ['Upside_Review', 'Downside_Review']
cols_score_sentences =['Upside_Review', 'Downside_Review', "text"]
# total_df.head()

In [22]:
english_df = total_df[total_df['Review_Language'] =='English'].copy()
english_df["Combined_Review"] = english_df["Upside_Review"].fillna("") + " " + english_df["Downside_Review"].fillna("")




In [23]:
# punctuation symbols and space
PUNC_SYS = regex.compile(r"[\p{P}\p{S}]")
SPACES = regex.compile(r"\s+")

def punct_clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = PUNC_SYS.sub(" ", text)
    text = SPACES.sub(" ", text)
    return text.strip()



In [24]:
LANG_MAP = {"English": "english", "French": "french", "German": "german",
            "Italian": "italian", "Spanish": "spanish", "Russian": "russian",
            "Romanian": "romanian", "Turkish": "turkish", "Chinese": "chinese"}
STOPWORDS = set(stopwords.words("english"))

def preprocessing_text(text):
    """
    Include stopword removal + tokenization
    """
    if not text:
        return []

    text = text.lower()
    tokens = text.split()
    #filtered_tokens = [word for word in tokens if word not in STOPWORDS]
    #return filtered_tokens
    return tokens

    

In [25]:
# preprocess engish df (fill empty reviews )

english_df['Combined_Review'] = english_df['Combined_Review'].map(punct_clean, na_action="ignore").fillna("")


english_df["Combined_Review"] = english_df["Combined_Review"].map(
    lambda t: " ".join(preprocessing_text(t))
)



In [26]:
english_df["Combined_Review"].iloc[0]

'staff serkan türk was a great help during our stay'

the following cell display the tokenizer for chinese only breaks on spaces not words

In [27]:
from sklearn.model_selection import StratifiedKFold, train_test_split


In [28]:
train_df, test_df= train_test_split(english_df, test_size = 0.25, stratify= english_df[['source', 'Sentiment']], random_state= 42)

In [29]:
X_train, y_train = train_df["Combined_Review"], train_df["source"]
X_test,  y_test  = test_df["Combined_Review"],  test_df["source"]
sentiment_test = test_df["Sentiment"]  #for q3 only

In [30]:
train_df[['source' , 'Sentiment']].value_counts()

source  Sentiment
1       NEG          375
        POS          375
0       POS          375
        NEG          375
Name: count, dtype: int64

In [31]:
len(X_train)


1500

In [32]:
len(X_test)

500

In [33]:
train_df[["source", "Sentiment"]].value_counts(normalize=True)

source  Sentiment
1       NEG          0.25
        POS          0.25
0       POS          0.25
        NEG          0.25
Name: proportion, dtype: float64

In [34]:
# save the splits
train_df.to_csv('train.csv', index = False)
test_df.to_csv('test.csv', index = False)